<a href="https://colab.research.google.com/github/DataSavvyYT/RAG-course/blob/main/01_rag_intro/dev/selecting_wines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets

In [2]:
import kagglehub
path = kagglehub.dataset_download("zynicide/wine-reviews")

100%|██████████| 50.9M/50.9M [00:00<00:00, 139MB/s]

Extracting files...


In [3]:
print(path)

/root/.cache/kagglehub/datasets/zynicide/wine-reviews/versions/4


In [5]:
import pandas as pd
import os

# List files in the downloaded directory to find the specific CSV
files = os.listdir(path)
print("Files in directory:", files)

# file is named 'winemag-data-130k-v2.csv'
csv_path = os.path.join(path, "winemag-data-130k-v2.csv")
df = pd.read_csv(csv_path)

# Display the first few rows
print(df.head())

Files in directory: ['winemag-data_first150k.csv', 'winemag-data-130k-v2.json', 'winemag-data-130k-v2.csv']
   Unnamed: 0   country                                        description  \
0           0     Italy  Aromas include tropical fruit, broom, brimston...   
1           1  Portugal  This is ripe and fruity, a wine that is smooth...   
2           2        US  Tart and snappy, the flavors of lime flesh and...   
3           3        US  Pineapple rind, lemon pith and orange blossom ...   
4           4        US  Much like the regular bottling from 2012, this...   

                          designation  points  price           province  \
0                        Vulkà Bianco      87    NaN  Sicily & Sardinia   
1                            Avidagos      87   15.0              Douro   
2                                 NaN      87   14.0             Oregon   
3                Reserve Late Harvest      87   13.0           Michigan   
4  Vintner's Reserve Wild Child Block      87   

In [15]:
df = df[df['variety'].notna()] # remove any NaN values as it blows up serialization
data = df.sample(700).to_dict('records') # Get only 700 records. More records will make it slower to index
len(data)

700

In [22]:
!pip install -q qdrant-client sentence-transformers fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.6 MB/s eta 0:00:00


In [9]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

In [10]:
encoder = SentenceTransformer('all-MiniLM-L6-v2') # Model to create embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
# create the vector database client
qdrant = QdrantClient(":memory:") # Create in-memory Qdrant instance

In [25]:
# Create collection to store wines
qdrant.recreate_collection(
    collection_name="top_wines",
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

/tmp/ipython-input-3075695827.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

In [26]:
# vectorize!
qdrant.upload_points(
    collection_name="top_wines",
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(doc["description"]).tolist(),
            payload=doc,
        ) for idx, doc in enumerate(data) # data is the variable holding all the wines
    ]
)

In [27]:
user_prompt = "Suggest me an amazing Malbec wine from Argentina"

In [38]:
query_vector = encoder.encode(user_prompt).tolist()

In [41]:
# Search time for awesome wines!

hits = qdrant.query_points(
    collection_name="top_wines",
    query=query_vector,
    limit=3
)

In [46]:
# Access the results via the .points attribute
for hit in hits.points:
    print(f"ID: {hit.id}, Score: {hit.score}, Payload: {hit.payload}")

ID: 59, Score: 0.6773974279539221, Payload: {'Unnamed: 0': 77560, 'country': 'US', 'description': "What a delicious barbecue wine. It shows Malbec's thick, sturdy tannins and California's ripe, sweet fruit, in the way of blackberries, cassis and blueberries, with a savory streak of espresso. But it's totally dry on the finish. Shows how well the Rockpile appellation ripens these big, full-bodied red wines.", 'designation': nan, 'points': 91, 'price': 42.0, 'province': 'California', 'region_1': 'Rockpile', 'region_2': 'Sonoma', 'taster_name': nan, 'taster_twitter_handle': nan, 'title': 'Keating 2008 Malbec (Rockpile)', 'variety': 'Malbec', 'winery': 'Keating'}
ID: 578, Score: 0.6491430083149315, Payload: {'Unnamed: 0': 129938, 'country': 'Argentina', 'description': "Compared to the regular 2006 Malbec from Chakana, this wine steps up in weight class and wins the crown. It's a serious but inviting red with spongey, ripe berry aromas topped by pure raspberry, cherry and cola flavors. It's

In [49]:
# define a variable to hold the search results
search_results = [hit.payload for hit in hits.points]

In [51]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

MessageError: TypeError: Failed to fetch

In [50]:
# Now time to connect to the local large language model
from openai import OpenAI
client = OpenAI(api_key=)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
completion = client.chat.completions.create(
    model="",
    messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": "Suggest me an amazing Malbec wine from Argentina"},
        {"role": "assistant", "content": str(search_results)}
    ]
)
print(completion.choices[0].message)